In [1]:
import random
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import dgl
from scipy.stats import entropy

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def seed_all(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
seed_all(42)

In [3]:
DATA = Path("../data/amazon_computers.pt")
OUT = Path("../outputs/")
OUT.mkdir(parents=True, exist_ok=True)

In [4]:
ck = torch.load(DATA, weights_only=False)
g, feats, labels = ck["graph"], ck["feats"], ck["labels"]
idx_tr, idx_va, idx_te = ck["idx_train"], ck["idx_val"], ck["idx_test"]


In [5]:
g = g.to(DEVICE)
feats = feats.to(DEVICE)
labels = labels.to(DEVICE)
N, d = feats.shape
C = int(labels.max() + 1)
t = torch.from_numpy(np.load(OUT / "teacher_logits.npz")["logits"]).to(DEVICE)
print(f"teacher {t.shape} N={N} E={g.num_edges()}")

teacher torch.Size([13752, 10]) N=13752 E=505474


In [6]:
def accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item()

In [7]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(d, 512)
        self.fc2 = nn.Linear(512, C)
        self.drop = nn.Dropout(0.5)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = self.drop(h)
        return self.fc2(h)
